# 02 · Base pipeline on `pilot_v1`: sketch + reference + text → image (UniCombine on FLUX)

This is the **aligned base** from the presentation, before any conflict handling:

| Presentation | What runs here |
|---|---|
| Text → T5/CLIP, joint attention | FLUX text branch (unchanged) |
| Sketch → OminiControl-style VAE-token injection + small LoRA | **Stage 1a:** train a `sketch` Condition-LoRA (UniCombine's single-condition mode *is* OminiControl) |
| Reference → LoRA-based FLUX customisation branch | **Stage 1b:** train a `reference` Condition-LoRA (appended tokens with shifted position ids, the way OminiControl handles subject images) |
| Fusion → UniCombine Conditional-MMDiT attention | **Stage 2:** train the union **Denoising-LoRA** on sketch+reference together |
| Evaluation plan | **Stage 4:** CLIP-T, CLIP-I/DINO-I, DreamSim, LPIPS, layout IoU, sketch recall, FID |
| Attribute Routing Layer | **not built yet** (Stage 5, needs this baseline first) |

**Ablation rows you get for free:** text-only · sketch-only · reference-only · both (no Denoising-LoRA) · both (trained).

### Read before running
- **Hardware.** The UniCombine trainer keeps the 12B FLUX transformer (~24 GB in bf16) *and* the T5-XXL/CLIP text encoders on the GPU, plus activations. My estimate is that you need a **60–80 GB GPU** (A100-80GB / H100). I expect this to fail on T4/L4/A100-40GB; that is exactly the "compute confirmation" from your slides, and the SDXL fallback would be the next thing to build.
- **Backbone.** UniCombine is built on **FLUX.1-schnell** (+ ostris' training adapter), *not* FLUX.1-dev. Keeping schnell is the path of least resistance; moving to dev means re-working the guidance-embedding path in the transformer. Your slides say the backbone is "subject to change", but it has to be written up as a deviation.
- **Fresh runtime.** This pins older library versions (the UniCombine code subclasses diffusers internals, and its last commit is June 2025). Do **not** reuse the runtime from the v2 notebook.
- **Pilot size.** ~150 training images is a smoke test of the pipeline, not a quality run. Expect weak sketch/reference following; scale the data after everything runs end to end.

In [1]:
import torch
p = torch.cuda.get_device_properties(0)
print(p.name, round(p.total_memory / 1e9), 'GB')
if p.total_memory < 60e9:
    print('STOP: under 60 GB. Training below will very likely run out of memory (see note above).')

Tesla T4 16 GB
STOP: under 60 GB. Training below will very likely run out of memory (see note above).


In [2]:
!pip install -q diffusers==0.33.1 transformers==4.52.4 peft==0.15.2 accelerate==1.7.0 optimum-quanto==0.2.7 sentencepiece protobuf opencv-python-headless datasets ipdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompati

Put the `pipeline/` folder (the `.py` files next to this notebook) in your Drive project folder, next to `pilot_v1.zip`. If a download asks for a Hugging Face token, run `from huggingface_hub import login; login()` first.

In [3]:
import os, glob, shutil, sys
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/sketchy_dataset_one_sketch'
UNI  = '/content/UniCombine'
DATA = '/content/pilot_v1'

!rm -rf {UNI} /content/pipeline {DATA}
!git clone -q https://github.com/Xuan-World/UniCombine.git {UNI}
!mkdir -p /content/pipeline && cp {ROOT}/pipeline/*.py /content/pipeline/
!cd /content/pipeline && python apply_patches.py {UNI} /content/pipeline
!unzip -q -o {ROOT}/pilot_v1.zip -d {DATA}
!cat {DATA}/manifest.json | head -20

Mounted at /content/drive
patch 4 applied to /usr/local/lib/python3.13/dist-packages/diffusers/loaders/peft.py
patched: train_cond_lora_sketchy.py, train_sketchy_union.py, src/condition.py
{
  "version": "pilot_v1",
  "frozen_at": "2026-10-21T07:32:35",
  "resolution": 512,
  "seed": 0,
  "split_unit": "photo",
  "counts": {
    "source_rows": 200,
    "kept": 200,
    "train": 160,
    "val": 20,
    "test": 20
  },
  "dropped_by_reason": {},
  "approved_fallback": [],
  "approved_failed": [],
  "manual_drop": [],
  "metadata_sha256": "65113436087c1f773e543a4e798c825e64e9a6888a6051397403c8f8ebc335ee",
  "sketch_polarity": "black strokes on white",
  "photo_to_square": "squash resize (matches how Sketchy sketches were drawn)",


In [7]:
from huggingface_hub import snapshot_download, login

# GATED REPO AUTHENTICATION STEP-BY-STEP:
# 1. Visit: https://huggingface.co/black-forest-labs/FLUX.1-schnell and accept terms.
# 2. Go to: https://huggingface.co/settings/tokens to generate a "Read" token.
# 3. Run this cell and paste your token in the login box.
try:
    login()
except Exception as e:
    print("Login failed or skipped. If the download fails, please authenticate with a valid Hugging Face token.")

# diffusers-format folders only (skips the 24 GB single-file copies)
snapshot_download('black-forest-labs/FLUX.1-schnell', local_dir=f'{UNI}/ckpt/FLUX.1-schnell',
                  ignore_patterns=['flux1-schnell.safetensors', 'ae.safetensors'])
snapshot_download('ostris/FLUX.1-schnell-training-adapter', local_dir=f'{UNI}/ckpt/FLUX.1-schnell-training-adapter')
print(os.listdir(f'{UNI}/ckpt/FLUX.1-schnell'), os.listdir(f'{UNI}/ckpt/FLUX.1-schnell-training-adapter'))

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

GatedRepoError: 403 Client Error. (Request ID: Root=1-6ad86dfd-7e2f20d15a9294cf1253975b;41d1bd38-3882-44f5-9a5e-1da1290753aa)

Cannot access gated repo for url https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/741f7c3ce8b383c54771c7003378a50191e9efe9/.gitattributes.
Access to model black-forest-labs/FLUX.1-schnell is restricted and you are not in the authorized list. Visit https://huggingface.co/black-forest-labs/FLUX.1-schnell to ask for access.

In [8]:
# sanity: the dataset class loads and yields the shapes the trainers expect
sys.path.insert(0, '/content/pipeline')
from sketchy_loader import SketchyPilot, collate_fn
ds = SketchyPilot(DATA, 'train', ('sketch', 'reference'))
b = collate_fn([ds[0], ds[1]])
print(len(ds), 'train samples |', b['pixel_values'].shape, b['condition_latents'].shape, b['condition_types'][0], '|', b['descriptions'][0][:80])

160 train samples | torch.Size([2, 3, 512, 512]) torch.Size([2, 2, 3, 512, 512]) ['sketch', 'reference'] | Blue-handled hammer with wooden handle.


## Stage 1: one Condition-LoRA per modality
Each trains on the same samples with only its own condition, so each LoRA learns what *that* modality is for (sketch → structure, reference → appearance). ~19 epochs at batch 1; raise `STEPS` once you have more data. Checkpoints land on local disk and the small LoRA files are copied to Drive at the end of each stage.

In [9]:
STEPS = 3000
!cd {UNI} && accelerate launch --num_processes 1 --mixed_precision bf16 train_cond_lora_sketchy.py \
    --output_condition_lora sketch --data_root {DATA} --work_dir /content/runs/sketch \
    --resolution 512 --train_batch_size 1 --max_train_steps {STEPS} --checkpointing_steps {STEPS // 3} --rank 4

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
I0000 00:00:1792568863.085300    3916 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792568865.406110    3916 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
INFO:__main__:Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: bf16

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_http.py", line 403, in

In [10]:
!cd {UNI} && accelerate launch --num_processes 1 --mixed_precision bf16 train_cond_lora_sketchy.py \
    --output_condition_lora reference --data_root {DATA} --work_dir /content/runs/reference \
    --resolution 512 --train_batch_size 1 --max_train_steps {STEPS} --checkpointing_steps {STEPS // 3} --rank 4

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
I0000 00:00:1792568889.355713    4050 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792568891.659078    4050 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
INFO:__main__:Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: bf16

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_http.py", line 403, in

In [11]:
def latest_ckpt(run):
    ck = glob.glob(f'{run}/*/checkpoint-*')
    return sorted(ck, key=lambda p: int(p.rsplit('-', 1)[1]))[-1]

os.makedirs(f'{UNI}/ckpt/Condition_LoRA', exist_ok=True)
for name in ['sketch', 'reference']:
    ck = latest_ckpt(f'/content/runs/{name}'); print(name, '<-', ck)
    shutil.copy(f'{ck}/{name}/pytorch_lora_weights.safetensors', f'{UNI}/ckpt/Condition_LoRA/{name}.safetensors')
shutil.copytree(f'{UNI}/ckpt/Condition_LoRA', f'{ROOT}/lora_pilot_v1/Condition_LoRA', dirs_exist_ok=True)

IndexError: list index out of range

## Stage 2: fusion (UniCombine Denoising-LoRA on sketch + reference)
The two Condition-LoRAs are loaded frozen; only the new Denoising-LoRA trains, with both conditions present. This is the "train on a multi-conditional dataset" step of UniCombine, and where the Conditional-MMDiT attention actually mixes the two condition streams.

In [ ]:
STEPS_UNION = 3000
!cd {UNI} && accelerate launch --num_processes 1 --mixed_precision bf16 train_sketchy_union.py \
    --condition_types sketch reference --pretrained_condition_lora_dir ckpt/Condition_LoRA \
    --output_denoising_lora sketch_reference_union --data_root {DATA} --work_dir /content/runs/union \
    --resolution 512 --train_batch_size 1 --max_train_steps {STEPS_UNION} --checkpointing_steps {STEPS_UNION // 3} --rank 4

In [ ]:
ck = latest_ckpt('/content/runs/union'); print(ck)
dst = f'{UNI}/ckpt/Denoising_LoRA/sketch_reference_union'
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(f'{ck}/sketch_reference_union', dst)
shutil.copytree(f'{UNI}/ckpt/Denoising_LoRA', f'{ROOT}/lora_pilot_v1/Denoising_LoRA', dirs_exist_ok=True)
print(os.listdir(dst))

## Stage 3: generate the test split in every mode
`set_mode` keeps the active adapters matched to the conditions passed in (a mismatch silently applies a condition LoRA to the image tokens). Generation is resumable per image.

In [ ]:
import unicombine_infer as UI
pipe = UI.load(UNI, f'{UNI}/ckpt/FLUX.1-schnell', f'{UNI}/ckpt/Condition_LoRA',
               f'{UNI}/ckpt/Denoising_LoRA', 'sketch_reference_union')
MODES = ['sketch', 'reference', 'both_free', 'both_trained']
for mode in MODES:
    UI.run_split(pipe, DATA, 'test', mode, f'/content/gen/{mode}')
print({m: len(os.listdir(f'/content/gen/{m}')) for m in MODES})

In [ ]:
# look at it: target | sketch | reference | sketch-only | reference-only | both (no Denoising-LoRA) | both (trained)
import json
from PIL import Image
from IPython.display import display
rows = [json.loads(l) for l in open(f'{DATA}/metadata.jsonl')]
test = [r for r in rows if r['split'] == 'test'][:5]
S = 200
sheet = Image.new('RGB', (S * 7, S * len(test)), 'white')
for i, r in enumerate(test):
    ims = [Image.open(f"{DATA}/{r[k]}") for k in ('target', 'sketch', 'reference')] + [Image.open(f"/content/gen/{m}/{r['id']}.png") for m in MODES]
    for j, im in enumerate(ims):
        sheet.paste(im.convert('RGB').resize((S, S)), (S * j, S * i))
display(sheet)
print([r['caption'][:70] for r in test])

### Text-only baseline
Plain FLUX.1-schnell with just the caption (no LoRAs, no conditions). This is the row that shows what the sketch and reference actually add.

In [ ]:
import gc
del pipe; gc.collect(); torch.cuda.empty_cache()
from diffusers import FluxPipeline
fp = FluxPipeline.from_pretrained(f'{UNI}/ckpt/FLUX.1-schnell', torch_dtype=torch.bfloat16).to('cuda')
os.makedirs('/content/gen/text_only', exist_ok=True)
for k, r in enumerate([r for r in rows if r['split'] == 'test']):
    out = f"/content/gen/text_only/{r['id']}.png"
    if not os.path.exists(out):
        fp(prompt=r['caption'], height=512, width=512, num_inference_steps=4, guidance_scale=0.0,
           max_sequence_length=256, generator=torch.Generator('cpu').manual_seed(k)).images[0].save(out)
del fp; gc.collect(); torch.cuda.empty_cache()

## Stage 4: metrics
Installs the metric libraries **after** training/generation (they can shift package versions). Per-image CSVs go to Drive. On a 15-20 image test split the numbers are only a wiring check, and FID is skipped below 50 images.

In [ ]:
!pip install -q lpips dreamsim "torchmetrics[image]" torch-fidelity

In [ ]:
import eval_metrics as EV, pandas as pd
os.makedirs(f'{ROOT}/eval_pilot_v1', exist_ok=True)
summ = {}
for mode in ['text_only'] + MODES:
    df_m, s = EV.evaluate_split(DATA, f'/content/gen/{mode}', 'test', out_csv=f'{ROOT}/eval_pilot_v1/{mode}.csv')
    summ[mode] = s
table = pd.DataFrame(summ).T
table.to_csv(f'{ROOT}/eval_pilot_v1/summary_aligned.csv')
table

How to read it (higher is better unless noted): `clip_t` caption match · `clip_i`/`dino_i` appearance similarity to the **reference** · `dreamsim`, `lpips` (**lower** is better) distance to the target photo · `layout_iou` object-mask overlap with the target · `sketch_recall` share of sketch strokes that land on an edge of the output, to compare with `sketch_recall_oracle` (the same number for the real photo, i.e. the ceiling). The thing to check is whether the adapters move these in the expected direction vs `text_only`: `sketch_recall`/`layout_iou` up with the sketch, `dino_i`/`clip_i` up with the reference, `clip_t` not collapsing.

## Stage 5 (next, not built): Attribute Routing Layer and the conflict subset
- **Where it goes:** `attn_forward` in `UniCombine/src/UniCombineTransformerBlock.py`. That is where each condition's keys/values are concatenated onto the image attention (`key = torch.cat([key, cond_key], dim=2)`). A routing layer gates or re-weights each condition stream per block/head, trained so sketch tokens dominate geometry, text dominates semantics, and reference tokens dominate appearance.
- **Conflict subset:** same `metadata.jsonl` format with `conflict_type` set to `text_reference`, `text_sketch` or `sketch_reference` and the mismatched asset paths swapped in. `unicombine_infer.run_split(..., meta_file='metadata_conflict.jsonl')` and `eval_metrics.evaluate_split(..., meta_file=...)` already take a custom metadata file.